In [ ]:
import time
notebook_start = time.perf_counter()
# %pip install -e /home/darshan/A6/PCSAFT_cDFT/thermoift

import os
import numpy  as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import thermoift.PLOT_SETTINGS as ps
import seaborn as sns
import feos


from thermoift.rng_utils                import get_rng
from thermoift                          import print_model_metrics
from thermoift.FeosPlugin               import (RegistryManager, ParameterBuilder, VLECalculator, CompositionHandler)
from sklearn.preprocessing              import StandardScaler
from sklearn.pipeline                   import Pipeline
from sklearn.gaussian_process           import GaussianProcessRegressor
from sklearn.gaussian_process.kernels   import Matern, ConstantKernel, WhiteKernel

RegistryManager.load_registry()


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG  (single reference temperature — composition-only AL)
# ═══════════════════════════════════════════════════════════════════════════════

POOL_PATH           = "../POOL/composition_pool.csv"
A4_PATH             = "../../DATASET_A4/CombinedDataset_A4.csv"
T_REF               = 220.0   # K — reference temperature for GPR hyperparameter fitting

# Sample sizes for learning-curve comparison
N_SIZES             = list(range(25, 125, 25))   # [25, 50, 75, 100]
N_MAX               = max(N_SIZES)               # 100

# Algorithm 1 parameters
N_t                 = None   # pool-evaluation budget per AL step (None → full P)
U_THRESHOLD         = 1e-6   # u_t: kernel-uncertainty cut-off (small → pure greedy)

# GPR fitting on A2 data
GPR_MAX_SAMPLES     = 5000
RESTART_OPTIMIZER   = 3
SEED                = 56852145
OUTPUT_DIR          = "AL_ST"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"N_SIZES  : {N_SIZES}")
print(f"T_REF    : {T_REF} K")
print(f"N_t      : {N_t}   (None = full P each step)")
print(f"u_t      : {U_THRESHOLD}")

N_SIZES  : [25, 50, 75, 100]
T_REF    : 220.0 K
N_t      : None   (None = full P each step)
u_t      : 1e-06


In [3]:
# ── Column-name bridge ────────────────────────────────────────────────────────
CSV_TO_FULL = {
    "CO2" : "carbon dioxide",
    "H2"  : "hydrogen",
    "Ar"  : "argon",
    "N2"  : "nitrogen",
    "CH4" : "methane",
    "O2"  : "oxygen",
    "CO"  : "carbon monoxide",
    "H2S" : "hydrogen sulfide",
}

POOL_Z_COLS = list(CSV_TO_FULL.keys())            # short names in pool CSV
Z_FEATURES  = [f"z_{v}" for v in CSV_TO_FULL.values()]  # long names in A2
COMP_MAP    = {k: f"z_{v}" for k, v in CSV_TO_FULL.items()}  # CO2→z_carbon dioxide…

# Composition-only: 8 mole fractions, no temperature feature
FEATURES    = Z_FEATURES
TARGET      = "P_bubble"
TARGET_UNIT = "bar"

print(f"Pool  columns : {POOL_Z_COLS}")
print(f"GPR features  : {FEATURES}  ({len(FEATURES)} total)")

Pool  columns : ['CO2', 'H2', 'Ar', 'N2', 'CH4', 'O2', 'CO', 'H2S']
GPR features  : ['z_carbon dioxide', 'z_hydrogen', 'z_argon', 'z_nitrogen', 'z_methane', 'z_oxygen', 'z_carbon monoxide', 'z_hydrogen sulfide']  (8 total)


In [4]:
# ── Load candidate pool (compositions only, 1000 rows) ────────────────────────
pool = pd.read_csv(POOL_PATH)
print(f"Pool loaded: {len(pool):,} compositions")
print(f"Columns: {list(pool.columns)}")
print()
print(pool.head(3))

Pool loaded: 1,000 compositions
Columns: ['feed_id', 'feed_source', 'template_name', 'mixture_size', 'active_components', 'CO2', 'H2', 'Ar', 'N2', 'CH4', 'O2', 'CO', 'H2S']

   feed_id         feed_source template_name  mixture_size  \
0        0  Random (Dirichlet)     dirichlet             4   
1        1  Random (Dirichlet)     dirichlet             3   
2        2  Random (Dirichlet)     dirichlet             6   

     active_components   CO2        H2        Ar       N2       CH4        O2  \
0        CO2,Ar,N2,H2S  0.97  0.000000  0.005316  0.00397  0.000000  0.000000   
1            CO2,Ar,O2  0.98  0.000000  0.000874  0.00000  0.000000  0.019126   
2  CO2,H2,Ar,CH4,O2,CO  0.95  0.007078  0.008244  0.00000  0.001661  0.029273   

         CO       H2S  
0  0.000000  0.020713  
1  0.000000  0.000000  
2  0.003745  0.000000  


In [5]:
# ── Load A2, filter to 8-component rows at T_REF, fit GPR ─────────────────────
a2          = pd.read_csv(A4_PATH)
a2.columns  = [c.strip() for c in a2.columns]

# Keep only rows where extra components (water, SO2, propane, ethane, etc.)
# contribute nothing — ensures z-vector sums to 1 for our 8 species
extra_z = [c for c in a2.columns if c.startswith("z_") and c not in Z_FEATURES]
if extra_z:
    a2 = a2[a2[extra_z].sum(axis=1) < 1e-6].copy()
    print(f"8-component rows: {len(a2):,}")

# Prefer rows at T_REF ± 1 K; fall back to all temperatures if too few
a2_ref = a2[np.abs(a2["temperature"] - T_REF) <= 1.0].copy()
if len(a2_ref) < 100:
    print(f"Only {len(a2_ref)} rows at T_REF — using all temperatures")
    a2_ref = a2.copy()

# Training set: composition → P_bubble at T_REF
df_train = (a2_ref[Z_FEATURES + [TARGET]]
            .drop_duplicates(subset=Z_FEATURES)
            .dropna(subset=[TARGET])
            .reset_index(drop=True))

if len(df_train) > GPR_MAX_SAMPLES:
    df_train = df_train.sample(n=GPR_MAX_SAMPLES, random_state=SEED).reset_index(drop=True)

print(f"GPR training set : {len(df_train):,} labeled A2 compositions  (T_REF={T_REF} K)")
print(f"P_bubble range   : {df_train[TARGET].min():.2f} – {df_train[TARGET].max():.2f} bar")

# ── Fit GPR: z → P_bubble(T_REF) ──────────────────────────────────────────────
n_feat = len(FEATURES)   # 8
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(nu=2.5, length_scale=np.ones(n_feat), length_scale_bounds=(1e-3, 1e3))
    + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 100.0))
)

gpr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("gpr",    GaussianProcessRegressor(
        kernel               = kernel,
        alpha                = 0.0,
        normalize_y          = True,
        n_restarts_optimizer = RESTART_OPTIMIZER,
        random_state         = SEED,
    )),
])

t0 = time.perf_counter()
gpr_model.fit(df_train[FEATURES], df_train[TARGET])
print(f"\nGPR fitting time : {time.perf_counter()-t0:.1f} s")

GPR training set : 99 labeled A2 compositions  (T_REF=220.0 K)
P_bubble range   : 7.22 – 109.51 bar

GPR fitting time : 0.6 s


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 6 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-pac

In [6]:
# ── GPR diagnostics and ARD feature importances ───────────────────────────────
gpr_step = gpr_model.named_steps["gpr"]
fitted_k = gpr_step.kernel_   # full fitted kernel

# Extract signal kernel (ConstantKernel × Matern) — used for AL uncertainty
# kernel_ structure: (ConstantKernel * Matern) + WhiteKernel → k1 + k2
signal_kernel = fitted_k.k1   # ConstantKernel × Matern  (no noise)

print("Fitted kernel:"); print(f"  {fitted_k}")
print()

# ARD length-scales → feature importances
ls   = signal_kernel.k2.length_scale          # one per feature
imp  = 1.0 / ls;  imp /= imp.sum()
print("ARD feature importances (1 / length-scale, normalised):")
for name, val in sorted(zip(FEATURES, imp), key=lambda x: -x[1]):
    bar = "█" * max(1, int(val * 40))
    print(f"  {name:<30s}  {val*100:5.1f}%  {bar}")

# Training-set fit quality
y_pred, _ = gpr_model.predict(df_train[FEATURES], return_std=True)
res = df_train[TARGET].values - y_pred
print(f"\nTrain RMSE : {np.sqrt(np.mean(res**2)):.3f} bar")
print(f"Train R²   : {1 - np.var(res)/np.var(df_train[TARGET].values):.4f}")


Fitted kernel:
  18.1**2 * Matern(length_scale=[155, 36.1, 1e+03, 369, 308, 1e+03, 1e+03, 121], nu=2.5) + WhiteKernel(noise_level=0.0001)

ARD feature importances (1 / length-scale, normalised):
  z_hydrogen                       53.9%  █████████████████████
  z_hydrogen sulfide               16.1%  ██████
  z_carbon dioxide                 12.6%  █████
  z_methane                         6.3%  ██
  z_nitrogen                        5.3%  ██
  z_argon                           1.9%  █
  z_oxygen                          1.9%  █
  z_carbon monoxide                 1.9%  █

Train RMSE : 0.029 bar
Train R²   : 1.0000


In [7]:
# ── ARD sensitivity to reference temperature ─────────────────────────────────
# Fits a composition-only GPR (Z_FEATURES, 8 features, no T) at each candidate
# temperature and compares the resulting ARD length-scales.

T_SENSITIVITY = [220, 230, 240, 250, 260, 270, 280, 300]  # K
T_WINDOW      = 2.0   # ± K window when filtering A2 rows per temperature
MIN_ROWS      = 50    # skip a temperature if fewer rows available

SHORT_NAMES = {
    "z_carbon dioxide"   : "CO2",
    "z_hydrogen"         : "H2",
    "z_argon"            : "Ar",
    "z_nitrogen"         : "N2",
    "z_methane"          : "CH4",
    "z_oxygen"           : "O2",
    "z_carbon monoxide"  : "CO",
    "z_hydrogen sulfide" : "H2S",
}

sensitivity_results = {}   # T → {'ls': array, 'imp': array, 'n_train': int}

print(f"{'T (K)':<8}  {'n_train':>8}  {'top component':<8}  {'l_min':>8}  {'l_max':>8}")
print("-" * 52)

for T in T_SENSITIVITY:
    mask = np.abs(a2["temperature"] - T) <= T_WINDOW
    df_T = (a2.loc[mask, Z_FEATURES + [TARGET]]
              .drop_duplicates(subset=Z_FEATURES)
              .dropna(subset=[TARGET])
              .reset_index(drop=True))

    if len(df_T) < MIN_ROWS:
        print(f"T={T} K: only {len(df_T)} rows — skipped")
        continue
    if len(df_T) > GPR_MAX_SAMPLES:
        df_T = df_T.sample(n=GPR_MAX_SAMPLES, random_state=SEED).reset_index(drop=True)

    # Composition-only GPR (8 features): sensitivity to T is assessed per-T
    k_T = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(nu=2.5, length_scale=np.ones(len(Z_FEATURES)),
                 length_scale_bounds=(1e-3, 1e3))
        + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-4, 100.0))
    )
    gpr_T = Pipeline([
        ("scaler", StandardScaler()),
        ("gpr", GaussianProcessRegressor(
            kernel=k_T, alpha=0.0, normalize_y=True,
            n_restarts_optimizer=RESTART_OPTIMIZER,
            random_state=SEED,
        )),
    ])
    gpr_T.fit(df_T[Z_FEATURES], df_T[TARGET])

    sig_k = gpr_T.named_steps["gpr"].kernel_.k1
    ls    = sig_k.k2.length_scale.copy()
    imp   = 1.0 / ls;  imp /= imp.sum()

    sensitivity_results[T] = {"ls": ls, "imp": imp, "n_train": len(df_T)}
    top = SHORT_NAMES[Z_FEATURES[int(np.argmin(ls))]]
    print(f"{T:<8}  {len(df_T):>8}  {top:<8}  {ls.min():>8.2f}  {ls.max():>8.1f}")

print(f"\nTemperatures successfully fitted: {sorted(sensitivity_results.keys())}")

T (K)      n_train  top component     l_min     l_max
----------------------------------------------------


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 6 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-pac

220             99  H2           36.14    1000.0


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 6 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-pac

230             99  H2           41.10    1000.0


...

/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-p

260             99  H2           74.14    1000.0


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 6 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-pac

270             99  H2           76.20    1000.0
280             99  H2           76.04    1000.0
T=300 K: only 1 rows — skipped

Temperatures successfully fitted: [220, 230, 240, 250, 260, 270, 280]


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/darshan/A6/py_A6/lib/python3.12/site-pac

In [ ]:
# ── Plot: ARD importance vs reference temperature ────────────────────────────

SHORT_NAMES = {
    "z_carbon dioxide"   : "CO2",
    "z_hydrogen"         : "H2",
    "z_argon"            : "Ar",
    "z_nitrogen"         : "N2",
    "z_methane"          : "CH4",
    "z_oxygen"           : "O2",
    "z_carbon monoxide"  : "CO",
    "z_hydrogen sulfide" : "H2S",
}

T_fitted   = sorted(sensitivity_results.keys())
imp_matrix = np.array([sensitivity_results[T]["imp"] for T in T_fitted])  # (n_T, 8)
ls_matrix  = np.array([sensitivity_results[T]["ls"]  for T in T_fitted])  # (n_T, 8)
short      = [SHORT_NAMES[f] for f in Z_FEATURES]                         # 8 names
imp_df     = pd.DataFrame(imp_matrix * 100, index=T_fitted, columns=short)
ls_df      = pd.DataFrame(ls_matrix,         index=T_fitted, columns=short)

colors = plt.cm.tab10(np.linspace(0, 1, len(Z_FEATURES)))

fig, ax = plt.subplots()
for j, (name, col) in enumerate(zip(short, colors)):
    ax.plot(T_fitted, imp_matrix[:, j] * 100,
            marker="o", markersize=5, linewidth=1.8, color=col, label=name)

ax.set_xlabel("Reference temperature (K)")
ax.set_ylabel(r"ARD importance (\%)")
ax.set_title("ARD feature importance vs reference temperature")
ax.legend(fontsize=7, ncol=2)
ps.apply_axis_style(ax)

out_dir = os.path.join(OUTPUT_DIR, "ARD_temperature_sensitivity")
os.makedirs(out_dir, exist_ok=True)
ps.save_plot(fig, "ARD_importance_vs_T", folder=out_dir)
plt.show()


In [9]:
# ── Scale pool features using fitted StandardScaler ───────────────────────────
# The GPR kernel was optimised on scaled features → apply same transform.
scaler = gpr_model.named_steps["scaler"]

pool_renamed = pool.rename(columns=COMP_MAP)   # CO2 → z_carbon dioxide …
X_pool_raw   = pool_renamed[FEATURES].values   # (1000, 8)  unscaled
X_pool       = scaler.transform(X_pool_raw)    # (1000, 8)  scaled

print(f"Pool feature matrix shape : {X_pool.shape}")
print(f"k(x,x) mean (first 5 rows): {signal_kernel.diag(X_pool[:5]).mean():.4f}  (≈ σ_f²)")

Pool feature matrix shape : (1000, 8)
k(x,x) mean (first 5 rows): 326.4410  (≈ σ_f²)


/home/darshan/A6/py_A6/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [10]:
# ── Algorithm 1 — kernel-based active learning (Xiang et al. 2023) ────────────

def compute_U_T(X_T, X_S, kernel):
    """
    Kernel uncertainty: U_T = diag(K_TT - K_TS K_SS⁻¹ K_ST)
    X_T : (|T|, d) — candidate batch
    X_S : (|S|, d) — already-selected set
    """
    K_TT_diag = kernel.diag(X_T)                      # (|T|,)
    K_TS      = kernel(X_T, X_S)                      # (|T|, |S|)
    K_SS      = kernel(X_S, X_S)                      # (|S|, |S|)
    K_SS_reg  = K_SS + 1e-8 * np.eye(len(X_S))
    try:
        L = np.linalg.cholesky(K_SS_reg)
        V = np.linalg.solve(L, K_TS.T)               # (|S|, |T|)
        U = K_TT_diag - np.einsum("ij,ij->j", V, V)
    except np.linalg.LinAlgError:
        K_inv = np.linalg.pinv(K_SS_reg)
        U = K_TT_diag - np.einsum("ij,jk,ki->i", K_TS, K_inv, K_TS.T)
    return np.maximum(U, 0.0)


def run_algorithm1(X_all, kernel, n_max, n_t=None, u_t=1e-6, seed=0):
    """
    Returns a list of pool indices in the order they were added to S.
    Starts with 2 random seeds; grows until |S| = n_max or P is empty.
    """
    rng     = np.random.default_rng(seed)
    n_pool  = len(X_all)
    idx_arr = np.arange(n_pool)

    # Initial S: 2 random points
    init    = rng.choice(n_pool, size=2, replace=False)
    s_list  = list(init)                  # ordered selection history
    p_set   = set(idx_arr) - set(s_list)  # remaining candidates

    while p_set and len(s_list) < n_max:
        p_list = list(p_set)

        # Optionally subsample candidates from P
        if n_t is not None and len(p_list) > n_t:
            t_idx  = rng.choice(len(p_list), size=n_t, replace=False)
            t_list = [p_list[i] for i in t_idx]
        else:
            t_list = p_list

        X_T = X_all[t_list]              # (|T|, d)
        X_S = X_all[s_list]              # (|S|, d)

        U = compute_U_T(X_T, X_S, kernel)  # (|T|,)

        # Add argmax to S if above threshold
        best_local = int(np.argmax(U))
        if U[best_local] > u_t:
            best_global = t_list[best_local]
            s_list.append(best_global)
            p_set.discard(best_global)

        # Remove "converged" points (U < u_t) from P → C (discard)
        for i, idx in enumerate(t_list):
            if U[i] < u_t:
                p_set.discard(idx)

    return s_list


print("Running Algorithm 1 …")
t0 = time.perf_counter()
al_order = run_algorithm1(
    X_all  = X_pool,         # (1000, 8)
    kernel = signal_kernel,
    n_max  = N_MAX,
    n_t    = N_t,
    u_t    = U_THRESHOLD,
    seed   = SEED,
)
elapsed_al = time.perf_counter() - t0
print(f"Algorithm 1 done: {len(al_order)} points selected in {elapsed_al:.1f} s")

# Build AL snapshots
al_selections = {}
for N in N_SIZES:
    n_avail = min(N, len(al_order))
    chosen  = list(al_order[:n_avail])
    if n_avail < N:
        fallback = [i for i in range(len(pool)) if i not in set(chosen)]
        rng_fb   = np.random.default_rng(SEED + N)
        rng_fb.shuffle(fallback)
        chosen = chosen + fallback[: N - n_avail]
    al_selections[N] = chosen

print("\nAL snapshot sizes:")
for N in N_SIZES:
    print(f"  N={N:3d}  selected={len(al_selections[N])}")

Running Algorithm 1 …
Algorithm 1 done: 87 points selected in 0.2 s

AL snapshot sizes:
  N= 25  selected=25
  N= 50  selected=50
  N= 75  selected=75
  N=100  selected=100


In [11]:
# ── Compute P_bubble(T_REF) for selected AL compositions via PC-SAFT ──────────
ALL_COMPS_FULL = [CSV_TO_FULL[k] for k in POOL_Z_COLS]

def compute_pbubble_single(z_arr, T_K):
    """PC-SAFT bubble-point. Returns P in bar, or NaN on failure."""
    active_z, active_comps, _ = CompositionHandler.reduce_components(
        z_arr, ALL_COMPS_FULL, verbose=False)
    params = ParameterBuilder.build_parameters(active_comps)
    func   = feos.HelmholtzEnergyFunctional.pcsaft(params)
    feed   = CompositionHandler.compute_feed_moles(active_z)
    T_bub, P_bub = VLECalculator.compute_bubble_curve(
        func, [T_K], feed, verbose=False)
    return float(P_bub[0]) if len(P_bub) else float('nan')


needed_idx = set()
for sel in al_selections.values():
    needed_idx.update(sel)
needed_idx = sorted(needed_idx)
print(f"Unique AL compositions to evaluate: {len(needed_idx)}")

pbubble_cache = {}   # pool_idx → P_bubble (bar)
n_failed = 0
t0 = time.perf_counter()

for i, idx in enumerate(needed_idx):
    row   = pool.iloc[idx]
    z_arr = np.array([row[k] for k in POOL_Z_COLS])
    try:
        p = compute_pbubble_single(z_arr, T_REF)
    except Exception as e:
        if n_failed < 3:
            print(f"  [warn] pool[{idx}]: {e}")
        p = float('nan')
        n_failed += 1
    pbubble_cache[idx] = p
    if (i + 1) % 20 == 0:
        print(f"  {i+1:4d}/{len(needed_idx)}  ({n_failed} failed so far) …")

elapsed_pb = time.perf_counter() - t0
n_ok = sum(1 for v in pbubble_cache.values() if not np.isnan(v))
print(f"\nP_bubble computed: {n_ok}/{len(needed_idx)} OK, {n_failed} failed")
print(f"Time: {elapsed_pb:.1f} s  ({elapsed_pb/max(len(needed_idx),1):.2f} s/composition)")

Unique AL compositions to evaluate: 100
    20/100  (0 failed so far) …
    40/100  (0 failed so far) …
    60/100  (0 failed so far) …
    80/100  (0 failed so far) …
   100/100  (0 failed so far) …

P_bubble computed: 100/100 OK, 0 failed
Time: 2.1 s  (0.02 s/composition)


In [12]:
# ── Assemble AL selection DataFrames (pool row + P_bubble column) ──────────────

def make_selection_df(idx_list, pool_df, pbubble_cache):
    rows = pool_df.iloc[idx_list].copy().reset_index(drop=True)
    rows['pool_idx'] = idx_list
    rows['P_bubble'] = [pbubble_cache.get(i, float('nan')) for i in idx_list]
    return rows

al_dfs = {}
for N in N_SIZES:
    al_dfs[N] = make_selection_df(al_selections[N], pool, pbubble_cache)

print("AL DataFrames built:")
for N, df in al_dfs.items():
    n_ok = df['P_bubble'].notna().sum()
    print(f"  N={N:3d}   P_bubble_ok={n_ok}/{N}")

AL DataFrames built:
  N= 25   P_bubble_ok=25/25
  N= 50   P_bubble_ok=50/50
  N= 75   P_bubble_ok=75/75
  N=100   P_bubble_ok=100/100


In [ ]:
# ── AL learning-curve plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: mean P_bubble vs N
ax = axes[0]
means = [al_dfs[N]['P_bubble'].mean() for N in N_SIZES]
ax.plot(N_SIZES, means, color="crimson", linestyle="-", linewidth=2,
        marker="o", markersize=5, label="AL")
ax.set_xlabel("Sample budget N")
ax.set_ylabel(f"Mean P_bubble at {T_REF} K (bar)")
ax.set_title("Mean bubble-point pressure vs sample size")
ax.legend(fontsize=9)
ps.apply_axis_style(ax)

# Right: CO2 diversity vs N
ax2 = axes[1]
stds = [al_dfs[N]['CO2'].std() for N in N_SIZES]
ax2.plot(N_SIZES, stds, color="crimson", linestyle="-", linewidth=2,
         marker="o", markersize=5, label="AL")
ax2.set_xlabel("Sample budget N")
ax2.set_ylabel("Std(CO2 fraction)")
ax2.set_title("CO2 fraction diversity vs sample size")
ax2.legend(fontsize=9)
ps.apply_axis_style(ax2)

plt.tight_layout()
ps.save_plot(fig, "AL_learning_curve", folder=OUTPUT_DIR)
plt.show()

# ── AL selection order in CO2 space (N=N_MAX) ─────────────────────────────────
fig2, ax3 = plt.subplots(figsize=(8, 4))
df_al_max = al_dfs[N_MAX]
ax3.scatter(range(len(df_al_max)), df_al_max["CO2"].values,
            c=range(len(df_al_max)), cmap="plasma", s=20, alpha=0.8)
ax3.set_xlabel("Selection order")
ax3.set_ylabel("CO2 fraction")
ax3.set_title(f"AL selection order (N={N_MAX}) in CO2 space")
plt.colorbar(ax3.collections[0], ax=ax3, label="Selection step")
ps.apply_axis_style(ax3)
plt.tight_layout()
ps.save_plot(fig2, "AL_selection_order_CO2", folder=OUTPUT_DIR)
plt.show()


In [14]:
# ── Save AL CSV files: one per N ─────────────────────────────────────────────
al_dir = os.path.join(OUTPUT_DIR, 'AL')
os.makedirs(al_dir, exist_ok=True)

for N, df in al_dfs.items():
    fname = os.path.join(al_dir, f'AL_N{N:03d}.csv')
    df.to_csv(fname, index=False)

print("Saved:")
for N in N_SIZES:
    print(f"  {os.path.join(al_dir, f'AL_N{N:03d}.csv')}")

# ── Save the fitted GPR model ──────────────────────────────────────────────────
model_path = os.path.join(OUTPUT_DIR, 'GPR_AL_model.joblib')
joblib.dump(gpr_model, model_path)
print(f"\nGPR model saved → {model_path}")

Saved:
  AL_ST/AL/AL_N025.csv
  AL_ST/AL/AL_N050.csv
  AL_ST/AL/AL_N075.csv
  AL_ST/AL/AL_N100.csv

GPR model saved → AL_ST/GPR_AL_model.joblib


In [15]:
elapsed_total = (time.perf_counter() - notebook_start) / 60
print(f"Total notebook runtime : {elapsed_total:.1f} min")
print(f"  Algorithm 1 (kernel) : {elapsed_al:.1f} s")
print(f"  P_bubble (PC-SAFT)   : {elapsed_pb:.1f} s")


Total notebook runtime : 0.4 min
  Algorithm 1 (kernel) : 0.2 s
  P_bubble (PC-SAFT)   : 2.1 s
